### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="micro_mass",
    dataset_year="2013",
    domain_str="biology & life sciences",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5T61S",
    download_description="""
wget https://archive.ics.uci.edu/static/public/253/micromass.zip && unzip micromass.zip micromass_un_anonymized.zip && rm micromass.zip && mkdir -p local-data-warehouse/micro_mass && unzip micromass_un_anonymized.zip -d local-data-warehouse/micro_mass/ && rm micromass_un_anonymized.zip
""",
    # References
    academic_reference_bibtex=r"""@article{mahe2014automatic,
  title={Automatic identification of mixed bacterial species fingerprints in a MALDI-TOF mass-spectrum},
  author={Mahe, Pierre and Arsac, Maud and Chatellier, Sonia and Monnin, Val{\'e}rie and Perrot, Nadine and Mailler, Sandrine and Girard, Victoria and Ramjeet, Mahendrasingh and Surre, J{\'e}r{\'e}my and Lacroix, Bruno and others},
  journal={Bioinformatics},
  volume={30},
  number={9},
  pages={1280--1286},
  year={2014},
  publisher={Oxford University Press}
}
""",
    academic_reference_bibtex_key="mahe2014automatic",
    license="CC BY 4.0",
    data_tags=["Non-IID", "Grouped"],
    curation_comments="""
We use the pure spectra version (without mixing) and replicate a grouped-based task to predict the species of strains.

- We drop columns with the same value across all samples.
- We drop constant columns (all 0).
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Species",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="Species",
    group_on="Strain",
    group_labels="per_group",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "micromass" / "pure_spectra_matrix.csv", header=None, names=[f"peak_list_spectra_bin_{i}" for i in range(1, 1301)], sep=";")
metadata = pd.read_csv(dataset_mold.path / "micromass" / "pure_spectra_metadata_with-species-name.csv", sep=";")
df = pd.concat([metadata, df], axis=1)
print("Loaded data shape:", df.shape)


as_cat_type = ["Strain", "Species"]
df[as_cat_type] = df[as_cat_type].astype("category")

# Drop duplicate columns based on values
df = df.loc[:, ~df.T.duplicated()]
# DRop constant columns
df = df.loc[:, (df != df.iloc[0]).any()]

df = df.sample(frac=1, random_state=42).sort_values(by="Strain").reset_index(drop=True)

## Data Checks

In [ ]:
df.groupby("Strain")["Species"].first().value_counts()

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
    group_labels=task_mold.group_labels,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits = curation_recommendations.get_recommended_grouped_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    group_on=task_mold.group_on,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    group_labels=task_mold.group_labels,
    show_splits=True,
    target_on=task_mold.target_column_name,
    random_state=split_random_state,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for grouped data.",
    splits=splits,
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()